## Without web search

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_deepseek import ChatDeepSeek

model = ChatDeepSeek(model="deepseek-chat")

In [3]:
from langchain.agents import create_agent

agent = create_agent(
    model=model
)

In [4]:
from langchain.messages import HumanMessage

question = HumanMessage(content="How up to date is your training knowledge?")

response = agent.invoke(
    {"messages": [question]}
)

In [5]:
print(response['messages'][-1].content)

My training data includes information up to **May 2025**. 

However, I don't have real-time browsing capabilities enabled by default in this chat session. If you need me to access current news, live stock prices, or the latest events that have occurred after May 2025, you can manually enable the **web search** or **browsing** feature in the interface. Once that's turned on, I can fetch the most recent information for you.

For general world knowledge and topics up to May 2025, I'm fully up to date! Let me know what you need.


## Add web search tool

In [15]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

web_search.invoke("Who is the current mayor of San Francisco?")

{'query': 'Who is the current mayor of San Francisco?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://en.wikipedia.org/wiki/Mayor_of_San_Francisco',
   'title': 'Mayor of San Francisco - Wikipedia',
   'content': 'Mayor of San Francisco ; Flag of San Francisco. Incumbent Daniel Lurie. since January 8, 2025. Government of San Francisco ; Flag of San Francisco. Incumbent',
   'score': 0.99854493,
   'raw_content': None},
  {'url': 'https://apnews.com/article/san-francisco-new-mayor-liberal-city-81ea0a7b37af6cbb68aea7ef5cc6a4f0',
   'title': "San Francisco's new mayor is starting to unite the fractured city",
   'content': 'San Francisco Mayor Daniel Lurie, a political newcomer and Levi Strauss heir, has marked his first 100 days with a hands-on, business-friendly approach.',
   'score': 0.9979966,
   'raw_content': None},
  {'url': 'https://www.sf.gov/departments--office-mayor',
   'title': 'Office of the Mayor - SF.gov',
   'content': 'Danie

In [7]:
agent = create_agent(
    model=model,
    tools=[web_search]
)

question = HumanMessage(content="Who is the current mayor of San Francisco?")

response = agent.invoke(
    {"messages": [question]}
)

In [8]:
from pprint import pprint

pprint(response['messages'])

[HumanMessage(content='Who is the current mayor of San Francisco?', additional_kwargs={}, response_metadata={}, id='ae116bee-6bad-4c00-84bb-d84e7f92c80a'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 280, 'total_tokens': 330, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256}, 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 24}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '6ad16a6d-1bea-4331-8008-1d4b2926118a', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e7982-456d-7d60-905b-fd237ba3ca50-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'current mayor of San Francisco 2024'}, 'id': 'call_00_Lo7m2GxJOtwEThtLeqNl8409', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 280, 'output_t

trace: https://smith.langchain.com/public/59432173-0dd6-49e8-9964-b16be6048426/r

In [9]:
from langchain.tools import tool
from ddgs import DDGS

@tool
def web_search_ddgs(query: str) -> str:
    """Search the web for information"""
    with DDGS() as ddgs:
        results = list(ddgs.text(query, max_results=5))
    return str(results)

In [10]:
agent = create_agent(
    model=model,
    tools=[web_search_ddgs]
)

question = HumanMessage(content="Who is the current mayor of San Francisco?")

response = agent.invoke(
    {"messages": [question]}
)

In [11]:
from pprint import pprint

pprint(response['messages'])

[HumanMessage(content='Who is the current mayor of San Francisco?', additional_kwargs={}, response_metadata={}, id='01a805b1-7afe-4b2f-b807-54129a2b3c9b'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 283, 'total_tokens': 333, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256}, 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 27}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '159d572b-9c83-4f66-baa9-b42dd9ca665c', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e7982-54f3-77f2-bea2-0a6f97446587-0', tool_calls=[{'name': 'web_search_ddgs', 'args': {'query': 'current mayor of San Francisco'}, 'id': 'call_00_G2pbet0zsNGhj8pKA8C92339', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 283, 'output_t